# 05 — Thực nghiệm và thử suy luận

**Notebook gốc:** bước 7–8. Bước 7 (GBT cross-day và random split) được gọi khi `WITH_SUPERVISED=1` trong notebook 03; code ở `src/baselines.py`. Notebook này đọc lại kết quả, đo **inference trên một batch pandas** và cho phép so sánh. **Đây không phải benchmark Spark end-to-end**, chỉ đo suy luận của mô hình trong một process.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Mở thư mục project trên VS Code, hoặc chạy notebook trong notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config as cfg
print("Project:", PROJECT_ROOT)
print("Dataset:", cfg.DATA_DIR)


Project: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode
Dataset: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode\data\raw


In [2]:
import json
import time
import joblib
import pandas as pd
import numpy as np
from IPython.display import display
from src.scoring import score_pandas

art_path = cfg.OUT_DIR / 'models' / 'model.joblib'
if not art_path.exists():
    raise FileNotFoundError('Chạy notebook 03 trước để tạo model.joblib')
art = joblib.load(art_path)
print('Số features:', len(art['cols']))
print('Ngưỡng:', art['thresholds'])


Số features: 46
Ngưỡng: {'s_iso': 0.5815025935353016, 's_ae': 2.5954462337493904, 's_ens': 5.3479444711721795}


In [3]:
# Đọc một phần nhỏ Parquet đã curated bằng PyArrow/Pandas (không cần Spark)
from pathlib import Path
parts = sorted(cfg.CURATED_DIR.glob('day=*/*.parquet'))
if not parts:
    raise FileNotFoundError('Không tìm thấy curated Parquet; chạy notebook 03 trước.')
# Lấy dữ liệu đúng schema; đọc file nhỏ nhất để hạn chế RAM
part = min(parts, key=lambda p: p.stat().st_size)
import pyarrow.parquet as pq
reader = pq.ParquetFile(part)
sample_df = next(reader.iter_batches(batch_size=1000, columns=art['cols'])).to_pandas()
print('Số dòng suy luận:', len(sample_df), '| file:', part.name)
t0 = time.perf_counter()
scored = score_pandas(sample_df, art)
seconds = time.perf_counter() - t0
print(f'{seconds:.3f}s cho {len(sample_df):,} flow (không gồm IO)')
display(scored.head(10))


Số dòng suy luận: 1000 | file: part-00153-06080c1c-c822-4b62-9c26-35372ef11b43.c000.snappy.parquet
0.020s cho 1,000 flow (không gồm IO)


,score,alert,top_feat
0,3.713729,False,active_std
1,-0.711988,False,is_tcp
2,5.128689,False,fwd_packet_length_min
3,-0.711987,False,is_tcp
4,3.315414,False,flow_packets_s
5,2.714719,False,flow_iat_std
6,6.283204,True,active_std
7,-0.711987,False,is_tcp
8,-0.711987,False,is_tcp
9,-0.711994,False,is_tcp


In [4]:
m = cfg.OUT_DIR / 'metrics' / 'metrics.json'
if m.exists():
    results = json.loads(m.read_text(encoding='utf-8'))
    print('Các tập kết quả đã lưu:')
    for key in results:
        print(' -', key)
    if not cfg.WITH_SUPERVISED:
        print('Bật WITH_SUPERVISED=1 trước khi chạy notebook 03 nếu muốn có GBT baseline.')


Các tập kết quả đã lưu:
 - fpr_target
 - test_frac
 - days
 - Isolation Forest
 - Autoencoder
 - Ensemble AE+IF
 - val_day_ensemble
 - recall_by_attack
 - top_features_of_detected_attacks
 - GBT cross-day (supervised, tham chiếu)
 - GBT random-split 70/15/15 (cận trên)


**Nâng cấp tiếp theo (chưa triển khai):** benchmark Spark end-to-end có đo đọc, chuyển đổi và ghi; ablation study; Kafka + Structured Streaming; Dashboard Streamlit. Không gọi các phần này đã hoàn thành.